# Token Economics Optimization for AI System Architects

## What you will build

You will build a classifier for a support desk that handles a high volume of tickets. Each ticket
arrives with a long history of customer messages and of logs from sub-tasks that already finished,
such as an account lookup. An orchestrator reads the ticket, three workers each decide one thing
about it, and a last step writes the routing decision.

Built the obvious way, every worker is sent the whole history, so each ticket pays for the same
text again and again. The diagram shows the three mistakes this course stops: paying for the
history at every handoff, a short summary that loses a fact a worker needed, and workers that wait
on each other when they could all run at once.

![What you will build](images/ticket-triage-overview.svg)

## Step 0: Set up the client and the model

Every call in this notebook goes through the repository's own client. Without an API key it plays
back responses recorded from real runs, so you can follow the whole course for free, and with a key
it calls the model live.

In [1]:
import math
import time
from concurrent.futures import ThreadPoolExecutor

from vault import Usage, get_client, load_env, model_for, summarise

load_env()
client = get_client("03-token-economics/01-classify-tickets-on-a-token-budget")
MODEL = model_for("default")

print(f"Client ready. Every request in this notebook uses {MODEL}.")

Client ready. Every request in this notebook uses google/gemini-2.5-flash-lite.


## Step 1: Create a support ticket with a long history

A classifier needs a realistic ticket to work on, so we start with one that has been open for a
day. Most of its history is the logs of **sub-tasks**, which are small jobs the desk ran on the
ticket, such as looking up the account or checking the charges.

In [2]:
def write_subtask_log(name, result, steps=20):
    """A sub-task's log: routine lines, then the one line that holds its result."""
    lines = [f"{name} step {i:02d}: called the {name} service, status 200, "
             f"{30 + i * 7} ms, cache miss, no retry" for i in range(1, steps + 1)]
    lines.append(f"{name} result: " + ", ".join(f"{k}={v}" for k, v in result.items()))
    return lines


def make_finished_subtask(name, result):
    """One finished sub-task as a history event, with its log and its result."""
    return {"kind": "subtask", "name": name, "status": "finished",
            "result": result, "log": write_subtask_log(name, result)}


print(write_subtask_log("account_lookup", {"customer_plan": "enterprise"}, steps=2))

['account_lookup step 01: called the account_lookup service, status 200, 37 ms, cache miss, no retry', 'account_lookup step 02: called the account_lookup service, status 200, 44 ms, cache miss, no retry', 'account_lookup result: customer_plan=enterprise']


The ticket is a duplicate charge on an invoice. Three sub-tasks have finished, one is still open,
and the customer has written twice.

In [3]:
TICKET_HISTORY = [
    {"kind": "customer", "text": "I was charged twice for this month's subscription "
                                 "on invoice INV-7731. Please refund the duplicate."},
    make_finished_subtask("account_lookup", {"customer_plan": "enterprise", "sla_hours": 4}),
    {"kind": "agent", "text": "Account found. Checking the charges on INV-7731 now."},
    make_finished_subtask("charge_check", {"duplicate_charge": "yes", "amount_cents": 49900}),
    make_finished_subtask("refund_eligibility", {"refund_eligible": "yes"}),
    {"kind": "customer", "text": "Any update? Our finance team needs this closed "
                                 "before the month ends."},
    {"kind": "subtask", "name": "refund_request", "status": "open", "result": {},
     "log": ["refund_request step 01: waiting for approval from the payments team"]},
]

`render_history` turns the events into the text a model reads, one event after another, with every
log line included.

In [4]:
def render_history(events):
    """The ticket history as the text a model reads."""
    lines = []
    for event in events:
        if event["kind"] == "subtask":
            lines.append(f"[sub-task {event['name']}, {event['status']}]")
            lines += event["log"]
        else:
            lines.append(f"{event['kind']}: {event['text']}")
    return "\n".join(lines)


FULL_HISTORY = render_history(TICKET_HISTORY)
print(f"{len(TICKET_HISTORY)} events, {len(FULL_HISTORY.splitlines())} lines, "
      f"{len(FULL_HISTORY)} characters")

7 events, 71 lines, 6682 characters


## Step 2: Classify the ticket with a chain that hands everything on

The first version is the one most teams write: each worker takes over the whole conversation so
far, adds its answer, and passes it on. Passing the work from one worker to the next is a
**handoff**, and this chain makes three of them before the synthesis step writes the routing line.

In [5]:
PIPELINE_PROMPT = ("You triage support tickets for a software company. "
                   "Answer only the question you are asked.")
WORKER_QUESTIONS = {
    "category": "Category worker: reply with one word, billing, technical, account or shipping.",
    "priority": "Priority worker: reply p1 if the customer plan is enterprise, p2 if it is pro, "
                "p3 if it is free. Reply with the label only.",
    "escalation": "Escalation worker: reply yes if the customer is angry or threatens to "
                  "leave, otherwise no. Reply with the word only.",
}
SYNTHESIS_QUESTION = ("Synthesis: write one routing line in the form "
                      "queue=<category>-<priority> escalate=<yes or no> note=<under 15 words>.")

print(f"{len(WORKER_QUESTIONS)} workers and a synthesis step: {list(WORKER_QUESTIONS)}")

3 workers and a synthesis step: ['category', 'priority', 'escalation']


`call_model` sends one request and returns three things: the reply, the usage the provider billed,
and how many seconds the call took. Every later step reads its costs and times from here.

In [6]:
MAX_REPLY_TOKENS = 200   # our own cap on each reply, not a provider limit


def call_model(messages):
    """One request. Returns the reply text, the billed usage and the seconds it took."""
    started = time.monotonic()
    response = client.chat.completions.create(model=MODEL, max_tokens=MAX_REPLY_TOKENS,
                                              temperature=0, messages=messages)
    seconds = time.monotonic() - started
    return (response.choices[0].message.content or "").strip(), Usage.from_response(response), seconds


print("call_model returns the reply, the usage and the seconds")

call_model returns the reply, the usage and the seconds


The next cell defines `classify_ticket_in_chain`, which is the chain itself. The history goes in
once at the top, and every question and answer after it stays in the list that each later call
sends.

In [7]:
def classify_ticket_in_chain(history_text):
    """Each worker takes over the whole conversation so far, then adds its own answer."""
    messages = [{"role": "system", "content": PIPELINE_PROMPT},
                {"role": "user", "content": f"Ticket history:\n{history_text}"}]
    answers, usages = {}, []
    for worker, question in [*WORKER_QUESTIONS.items(), ("synthesis", SYNTHESIS_QUESTION)]:
        messages.append({"role": "user", "content": question})
        answer, usage, _ = call_model(messages)
        messages.append({"role": "assistant", "content": answer})
        answers[worker] = answer
        usages.append(usage)
        print(f"{worker:10} prompt tokens {usage.prompt_tokens:5}   answer {answer!r}")
    return answers, usages


chain_answers, chain_usages = classify_ticket_in_chain(FULL_HISTORY)

category   prompt tokens  2203   answer 'billing'


priority   prompt tokens  2237   answer 'p1'


escalation prompt tokens  2265   answer 'no'


synthesis  prompt tokens  2296   answer 'queue=billing-p1 escalate=no note=Duplicate charge on INV-7731, refund eligible.'


## Step 3: Measure the token tax that every handoff pays

A **token** is the unit a model reads and bills in, roughly a short word or part of one, and the
provider bills every token a request sends. Everything one call sends is its **context payload**,
so the next cell compares the payload of the first call with the prompt tokens the whole chain paid.

In [8]:
first_payload = chain_usages[0].prompt_tokens
handoffs = len(chain_usages) - 1
prompt_bill = summarise(chain_usages)["prompt_tokens"]
token_tax = first_payload * handoffs

print(f"first call's context payload : {first_payload} tokens")
print(f"prompt tokens for the chain  : {prompt_bill} tokens over {len(chain_usages)} calls")
print(f"first payload resent         : {handoffs} times, at least {token_tax} tokens")
print(f"share of the prompt bill     : {token_tax / prompt_bill:.0%}")

first call's context payload : 2203 tokens
prompt tokens for the chain  : 9001 tokens over 4 calls
first payload resent         : 3 times, at least 6609 tokens
share of the prompt bill     : 73%


The first call sent 2203 tokens, and almost all of them were the ticket history. Each of the three
later calls sent all of that again, plus the questions and answers before it, so at least 6609 of
the chain's 9001 prompt tokens paid for text an earlier call had already sent. That repeat payment
is the **token tax** of a handoff, and it grows with every worker you add to the chain.

The answers themselves are right, including a priority of p1, because the account lookup's log
says the customer is on the enterprise plan. Any cheaper version has to keep that answer.

## Step 4: Hand each worker a key-value state instead of the history

The first fix stops the history from travelling with every handoff. One call, the **orchestrator**,
reads the history once and writes the ticket's state as a few `key: value` lines, and each worker
then reads only those lines.

![Hand each worker a key-value state instead of the history](images/ticket-state-step-1.svg)

In [9]:
ORCHESTRATOR_PROMPT = ("Read the support ticket history. Write its state as key: value lines, "
                       "one per line, with exactly these keys: {keys}. "
                       "Write unknown for any value the history does not give.")
STATE_KEYS = ["issue", "customer_request", "customer_tone", "customer_plan"]


def parse_key_values(text):
    """Turn 'key: value' lines into a dict. Lines without a colon are ignored."""
    state = {}
    for line in text.splitlines():
        key, colon, value = line.partition(":")
        if colon:
            state[key.strip(" -*").lower()] = value.strip(" *")
    return state


def format_state(state):
    """The state as the key: value lines a worker reads."""
    return "\n".join(f"{key}: {value}" for key, value in state.items())


print(parse_key_values("issue: duplicate charge\ncustomer_plan: enterprise"))

{'issue': 'duplicate charge', 'customer_plan': 'enterprise'}


Each of the three calls below reads a small payload. Only `summarise_ticket_state` ever sees the
ticket history.

In [10]:
def summarise_ticket_state(history_text, keys):
    """The orchestrator: the only call that reads the ticket history."""
    prompt = ORCHESTRATOR_PROMPT.format(keys=", ".join(keys))
    reply, usage, seconds = call_model([{"role": "system", "content": prompt},
                                        {"role": "user", "content": history_text}])
    return parse_key_values(reply), usage, seconds


def run_worker(worker, state):
    """One worker reads the small state and answers its one question."""
    content = f"Ticket state:\n{format_state(state)}\n\n{WORKER_QUESTIONS[worker]}"
    return call_model([{"role": "system", "content": PIPELINE_PROMPT},
                       {"role": "user", "content": content}])


def synthesise_routing(state, labels):
    """The last call turns the state and the worker labels into one routing line."""
    content = f"Ticket state:\n{format_state({**state, **labels})}\n\n{SYNTHESIS_QUESTION}"
    return call_model([{"role": "system", "content": PIPELINE_PROMPT},
                       {"role": "user", "content": content}])


print("orchestrator, worker and synthesis calls defined")

orchestrator, worker and synthesis calls defined


`triage_ticket` joins them up and records the usage and the seconds of every call, because the
later steps compare both.

In [11]:
def run_workers_in_sequence(state):
    """Run each worker after the one before it has finished."""
    return {worker: run_worker(worker, state) for worker in WORKER_QUESTIONS}


def triage_ticket(history_text, keys, known_state=None, run_workers=run_workers_in_sequence):
    """Orchestrator, then the workers, then synthesis. Records what every call cost and took."""
    started = time.monotonic()
    state, usage, seconds = summarise_ticket_state(history_text, keys)
    state.update(known_state or {})
    calls = {"orchestrator": (usage, seconds)}
    workers = run_workers(state)
    labels = {worker: answer for worker, (answer, _, _) in workers.items()}
    calls.update({worker: (usage, seconds) for worker, (_, usage, seconds) in workers.items()})
    routing, usage, seconds = synthesise_routing(state, labels)
    calls["synthesis"] = (usage, seconds)
    return {"state": state, "labels": labels, "routing": routing, "calls": calls,
            "wall_seconds": time.monotonic() - started}


print("triage_ticket runs the orchestrator, the workers and synthesis")

triage_ticket runs the orchestrator, the workers and synthesis


`print_triage_report` prints each call's payload and time, then the decision the pipeline reached.

In [12]:
def print_triage_report(run):
    """Each call's context payload and time, then what the pipeline decided."""
    for name, (usage, seconds) in run["calls"].items():
        print(f"{name:12} prompt tokens {usage.prompt_tokens:5}   {seconds:4.2f} s")
    print(f"state        : {run['state']}")
    print(f"labels       : {run['labels']}")
    print(f"routing      : {run['routing']}")


state_run = triage_ticket(FULL_HISTORY, STATE_KEYS)
print_triage_report(state_run)

orchestrator prompt tokens  2214   0.67 s
category     prompt tokens    74   0.38 s
priority     prompt tokens    91   0.48 s
escalation   prompt tokens    84   0.31 s
synthesis    prompt tokens   102   0.46 s
state        : {'issue': 'Duplicate charge on invoice INV-7731', 'customer_request': 'Refund the duplicate charge', 'customer_tone': 'Urgent, slightly anxious', 'customer_plan': 'unknown'}
labels       : {'category': 'billing', 'priority': 'p2', 'escalation': 'no'}
routing      : queue=billing-p2 escalate=no note=Customer reports duplicate charge on invoice INV-7731.


The workers' payloads fell from over 2200 tokens each to between 74 and 102, and only the
orchestrator still reads the history. The priority, though, changed from p1 to p2, which sends an
enterprise customer to a slower queue.

The state says why. The orchestrator wrote `customer_plan: unknown`, although the account lookup's
last log line says `customer_plan=enterprise`. That one line sat under twenty routine log lines, the
summary missed it, and the priority worker guessed instead of stopping.

## Step 5: Prune finished sub-task logs before the orchestrator reads them

The orchestrator still pays for every log line of every finished sub-task. **Context pruning** means
removing text from a payload that no later step needs, and a finished sub-task's log looks like
exactly that kind of text.

![Prune finished sub-task logs before the orchestrator reads them](images/ticket-state-step-2.svg)

In [13]:
def prune_finished_subtasks(events):
    """Drop every finished sub-task from the history, log and all."""
    return [event for event in events
            if not (event["kind"] == "subtask" and event["status"] == "finished")]


PRUNED_HISTORY = render_history(prune_finished_subtasks(TICKET_HISTORY))
print(f"history before pruning: {len(FULL_HISTORY.splitlines())} lines, {len(FULL_HISTORY)} characters")
print(f"history after pruning : {len(PRUNED_HISTORY.splitlines())} lines, "
      f"{len(PRUNED_HISTORY)} characters")

history before pruning: 71 lines, 6682 characters
history after pruning : 5 lines, 349 characters


The open sub-task stays, because its log may still matter. The next cell runs the same pipeline on
the pruned history.

In [14]:
pruned_run = triage_ticket(PRUNED_HISTORY, STATE_KEYS)
print_triage_report(pruned_run)

orchestrator prompt tokens   137   0.79 s
category     prompt tokens    74   0.63 s
priority     prompt tokens    91   0.37 s
escalation   prompt tokens    84   0.33 s
synthesis    prompt tokens   102   0.49 s
state        : {'issue': 'Duplicate charge on invoice INV-7731', 'customer_request': 'Refund for duplicate charge', 'customer_tone': 'Urgent, slightly impatient', 'customer_plan': 'unknown'}
labels       : {'category': 'billing', 'priority': 'p2', 'escalation': 'no'}
routing      : queue=billing-p2 escalate=no note=Duplicate charge on invoice INV-7731.


Pruning cut the history from 71 lines to 5, and the orchestrator's payload from 2214 tokens to 137.
The priority is still p2, and now the plan could not have been found at all, because the only line
that held it was dropped along with the rest of the account lookup's log.

## Step 6: Keep each finished result when its log is pruned

A finished sub-task's log is noise, but its result is a fact the workers need. The fix keeps the
result as state in code, and adds a check that refuses to start a worker whose keys are missing,
so no worker is paid to guess.

![Keep each finished result when its log is pruned](images/ticket-state-step-3.svg)

In [15]:
WORKER_NEEDS = {"category": ["issue"], "priority": ["customer_plan"],
                "escalation": ["customer_tone"]}


class MissingStateError(Exception):
    """A worker would have to guess, because the state lacks a key it needs."""


def check_state_for_workers(state):
    """Refuse to start any worker whose keys are missing or unknown."""
    for worker, keys in WORKER_NEEDS.items():
        missing = [key for key in keys if state.get(key, "unknown").lower() in ("", "unknown")]
        if missing:
            raise MissingStateError(f"the {worker} worker needs {missing}, "
                                    f"and the state does not hold it")


try:
    check_state_for_workers(pruned_run["state"])
except MissingStateError as error:
    print(f"refused before any worker ran: {error}")

refused before any worker ran: the priority worker needs ['customer_plan'], and the state does not hold it


`prune_finished_subtasks_keeping_results` drops each finished log exactly as before, but first
copies its result into a dict. Those results are facts our code already holds, so they go straight
into the state and never pass through a model.

In [16]:
def prune_finished_subtasks_keeping_results(events):
    """Drop each finished sub-task's log, but keep its result as key-value state."""
    results = {}
    for event in events:
        if event["kind"] == "subtask" and event["status"] == "finished":
            results.update(event["result"])
    return prune_finished_subtasks(events), results


def run_workers_after_check(state):
    """Check the state first, then run the workers one after another."""
    check_state_for_workers(state)
    return run_workers_in_sequence(state)


kept_events, SUBTASK_RESULTS = prune_finished_subtasks_keeping_results(TICKET_HISTORY)
print(f"results kept from finished sub-tasks: {SUBTASK_RESULTS}")

results kept from finished sub-tasks: {'customer_plan': 'enterprise', 'sla_hours': 4, 'duplicate_charge': 'yes', 'amount_cents': 49900, 'refund_eligible': 'yes'}


The orchestrator is now asked only for what the conversation holds, and the results from code fill
in the rest of the state.

In [17]:
CONVERSATION_KEYS = ["issue", "customer_request", "customer_tone"]

lean_run = triage_ticket(render_history(kept_events), CONVERSATION_KEYS,
                         known_state=SUBTASK_RESULTS, run_workers=run_workers_after_check)
print_triage_report(lean_run)

orchestrator prompt tokens   133   0.54 s
category     prompt tokens   101   0.32 s
priority     prompt tokens   118   0.41 s
escalation   prompt tokens   111   0.33 s
synthesis    prompt tokens   129   0.58 s
state        : {'issue': 'Duplicate charge on invoice INV-7731', 'customer_request': 'Refund for duplicate charge', 'customer_tone': 'Urgent', 'customer_plan': 'enterprise', 'sla_hours': 4, 'duplicate_charge': 'yes', 'amount_cents': 49900, 'refund_eligible': 'yes'}
labels       : {'category': 'billing', 'priority': 'p1', 'escalation': 'no'}
routing      : queue=billing-p1 escalate=no note=Duplicate charge on invoice INV-7731.


The priority is back to p1 and the routing line names the right queue, while the orchestrator's
payload stays at 133 tokens. The plan now comes from the account lookup's result, which our code
kept when it dropped the log, so no summary can lose it. If a result is ever missing, the check
refuses the ticket, as it refused the state from the previous step.

## Step 7: Compare the cost of one ticket before and after

The desk pays per ticket, so the unit that matters is the cost of one classified ticket, not the
cost of one call. The next cell prices all three versions with the provider's own rates and scales
each one up to a day of tickets.

In [18]:
TICKETS_PER_DAY = 20_000   # the desk's volume, used only to scale one ticket up


def list_usages(run):
    """Every usage one triage run was billed for."""
    return [usage for usage, _ in run["calls"].values()]


VERSIONS = [("chain with the whole history", chain_usages),
            ("state, logs not pruned", list_usages(state_run)),
            ("state, logs pruned, results kept", list_usages(lean_run))]

for label, usages in VERSIONS:
    bill = summarise(usages)
    print(f"{label:33} {bill['calls']} calls {bill['prompt_tokens']:5} prompt tokens "
          f"${bill['usd']:.6f} per ticket, ${bill['usd'] * TICKETS_PER_DAY:.2f} per day")

chain with the whole history      4 calls  9001 prompt tokens $0.000911 per ticket, $18.23 per day
state, logs not pruned            5 calls  2565 prompt tokens $0.000282 per ticket, $5.64 per day
state, logs pruned, results kept  5 calls   592 prompt tokens $0.000080 per ticket, $1.61 per day


Handing on a key-value state cut the prompt tokens for one ticket from 9001 to 2565, and pruning
the finished logs cut them again to 592. In this run, that is the difference between $18.23 and
$1.61 a day at the desk's volume.

The middle row needs care, because it was cheaper than the chain and it routed the ticket to the
wrong priority. A saving only counts once the answer has been checked against the version it
replaces.

## Step 8: Run the three workers in parallel

Now that each worker reads only the state, no worker waits for another worker's answer, so all
three can run at once. `run_workers_in_parallel` checks the state, starts every worker in its own
thread, and waits until the last one finishes.

![Run the three workers in parallel](images/worker-fanout-step-1.svg)

In [19]:
def run_workers_in_parallel(state):
    """Check the state, then start every worker at once and wait for all of them."""
    check_state_for_workers(state)
    with ThreadPoolExecutor(max_workers=len(WORKER_QUESTIONS)) as pool:
        futures = {worker: pool.submit(run_worker, worker, state) for worker in WORKER_QUESTIONS}
        return {worker: future.result() for worker, future in futures.items()}


parallel_run = triage_ticket(PRUNED_HISTORY, CONVERSATION_KEYS,
                             known_state=SUBTASK_RESULTS, run_workers=run_workers_in_parallel)
print_triage_report(parallel_run)
print(f"\nwall clock, workers in sequence: {lean_run['wall_seconds']:.2f} s")
print(f"wall clock, workers in parallel: {parallel_run['wall_seconds']:.2f} s")

orchestrator prompt tokens   133   0.47 s
category     prompt tokens   101   0.57 s
priority     prompt tokens   118   0.84 s
escalation   prompt tokens   111   0.62 s
synthesis    prompt tokens   129   0.44 s
state        : {'issue': 'Duplicate charge on invoice INV-7731', 'customer_request': 'Refund for duplicate charge', 'customer_tone': 'Urgent', 'customer_plan': 'enterprise', 'sla_hours': 4, 'duplicate_charge': 'yes', 'amount_cents': 49900, 'refund_eligible': 'yes'}
labels       : {'category': 'billing', 'priority': 'p1', 'escalation': 'no'}
routing      : queue=billing-p1 escalate=no note=Duplicate charge on invoice INV-7731.

wall clock, workers in sequence: 2.17 s
wall clock, workers in parallel: 1.76 s


Running the workers at once cut the wall clock from 2.17 to 1.76 seconds, with the same labels and
the same routing line. Each worker call also took longer when three ran together, 0.84 seconds at
worst against 0.41 in sequence, so the saving is smaller than the sequential times suggest.

## Step 9: Predict the wall clock with latency math

A pipeline's wall clock can be worked out from the time of each call. That arithmetic is **latency
math**: in sequence the times add up, and in parallel the wall clock is the orchestrator plus the
slowest worker plus synthesis.

![Predict the wall clock with latency math](images/worker-fanout-step-2.svg)

In [20]:
def predict_sequential_seconds(seconds):
    """Every call waits for the one before it, so the times add up."""
    return sum(seconds.values())


def predict_parallel_seconds(seconds):
    """The orchestrator, then the slowest worker, then synthesis."""
    slowest = max(seconds[worker] for worker in WORKER_QUESTIONS)
    return seconds["orchestrator"] + slowest + seconds["synthesis"]


print("latency math: sequence adds every call, parallel adds only the slowest worker")

latency math: sequence adds every call, parallel adds only the slowest worker


The next cell applies each prediction to the call times a run recorded, and puts it beside the wall
clock measured for that run.

In [21]:
for label, run, predict in [("in sequence", lean_run, predict_sequential_seconds),
                            ("in parallel", parallel_run, predict_parallel_seconds)]:
    seconds = {name: took for name, (_, took) in run["calls"].items()}
    slowest = max(WORKER_QUESTIONS, key=seconds.get)
    print(f"workers {label}: predicted {predict(seconds):.2f} s, "
          f"measured {run['wall_seconds']:.2f} s, slowest worker {slowest} "
          f"at {seconds[slowest]:.2f} s")

workers in sequence: predicted 2.17 s, measured 2.17 s, slowest worker priority at 0.41 s
workers in parallel: predicted 1.76 s, measured 1.76 s, slowest worker priority at 0.84 s


Both predictions land within a hundredth of a second of the measured wall clock. In parallel, the
priority worker at 0.84 seconds set the pace, so making the other two workers faster would not have
changed the wall clock at all. To cut it further, you shorten the orchestrator, the slowest worker
or synthesis, because nothing else appears in the sum.

## Step 10: Test the pipeline without calling the model

Each piece above gets a test that runs in milliseconds with no API key, so it can run on every
commit. If someone stops keeping the results, drops the state check, or changes how the workers are
joined, one of these tests fails.

![Test the pipeline without calling the model](images/worker-fanout-step-3.svg)

In [22]:
def test_pruned_history_stays_flat_as_work_finishes():
    extra = [make_finished_subtask(f"extra_check_{i}", {f"extra_{i}": "done"}) for i in range(5)]
    events, results = prune_finished_subtasks_keeping_results(TICKET_HISTORY + extra)
    assert render_history(events) == PRUNED_HISTORY
    assert all(f"extra_{i}" in results for i in range(5))


def test_worker_without_its_state_is_refused():
    state = {"issue": "duplicate charge", "customer_tone": "calm", "customer_plan": "unknown"}
    try:
        check_state_for_workers(state)
    except MissingStateError:
        return
    raise AssertionError("the priority worker was allowed to guess the plan")

The last test pins the latency math to a case worked out by hand, where the priority worker is the
slowest of the three.

In [23]:
def test_parallel_time_is_orchestrator_plus_slowest_plus_synthesis():
    seconds = {"orchestrator": 1.0, "category": 0.4, "priority": 0.9,
               "escalation": 0.5, "synthesis": 0.6}
    assert math.isclose(predict_parallel_seconds(seconds), 2.5)
    assert math.isclose(predict_sequential_seconds(seconds), 3.4)


for test in (test_pruned_history_stays_flat_as_work_finishes,
             test_worker_without_its_state_is_refused,
             test_parallel_time_is_orchestrator_plus_slowest_plus_synthesis):
    test()
    print(f"passed: {test.__name__}")

passed: test_pruned_history_stays_flat_as_work_finishes
passed: test_worker_without_its_state_is_refused
passed: test_parallel_time_is_orchestrator_plus_slowest_plus_synthesis


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Handoff** | `classify_ticket_in_chain` | Passes the work from one worker to the next, with everything sent so far |
| **Token tax** | the first payload times the handoffs | The prompt tokens paid again at every handoff for text already sent |
| **Context payload** | `usage.prompt_tokens` | Everything one call sends, billed at the prompt rate |
| **Key-value state** | `summarise_ticket_state` and `format_state` | A few lines the workers read instead of the ticket history |
| **Context pruning** | `prune_finished_subtasks_keeping_results` | Drops finished logs from the payload and keeps their results |
| **State check** | `check_state_for_workers` | Refuses to start a worker whose keys are missing, so it never guesses |
| **Cost per ticket** | `summarise` over one run's usages | The unit the desk actually pays for |
| **Latency math** | `predict_parallel_seconds` | Orchestrator plus the slowest worker plus synthesis |